In [1]:
!pip install xgboost

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import accuracy_score
from xgboost import XGBClassifier

In [2]:
df = pd.read_csv("synthetic_heart_disease_dataset.csv")
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    50000 non-null  int64  
 1   Gender                 50000 non-null  object 
 2   Weight                 50000 non-null  int64  
 3   Height                 50000 non-null  int64  
 4   BMI                    50000 non-null  float64
 5   Smoking                50000 non-null  object 
 6   Alcohol_Intake         29891 non-null  object 
 7   Physical_Activity      50000 non-null  object 
 8   Diet                   50000 non-null  object 
 9   Stress_Level           50000 non-null  object 
 10  Hypertension           50000 non-null  int64  
 11  Diabetes               50000 non-null  int64  
 12  Hyperlipidemia         50000 non-null  int64  
 13  Family_History         50000 non-null  int64  
 14  Previous_Heart_Attack  50000 non-null  int64  
 15  Sy

In [3]:
df.isnull().sum()

Age                          0
Gender                       0
Weight                       0
Height                       0
BMI                          0
Smoking                      0
Alcohol_Intake           20109
Physical_Activity            0
Diet                         0
Stress_Level                 0
Hypertension                 0
Diabetes                     0
Hyperlipidemia               0
Family_History               0
Previous_Heart_Attack        0
Systolic_BP                  0
Diastolic_BP                 0
Heart_Rate                   0
Blood_Sugar_Fasting          0
Cholesterol_Total            0
Heart_Disease                0
dtype: int64

In [4]:
df.drop(columns="Alcohol_Intake", inplace=True)

In [5]:
df.head()


,Age,Gender,Weight,Height,BMI,Smoking,Physical_Activity,Diet,Stress_Level,Hypertension,Diabetes,Hyperlipidemia,Family_History,Previous_Heart_Attack,Systolic_BP,Diastolic_BP,Heart_Rate,Blood_Sugar_Fasting,Cholesterol_Total,Heart_Disease
0,48,Male,78,157,26.4,Never,Sedentary,Healthy,Medium,0,0,1,1,0,104,99,71,165,200,0
1,35,Female,73,163,33.0,Never,Active,Average,High,1,0,1,1,0,111,72,60,145,206,0
2,79,Female,88,152,32.3,Never,Moderate,Average,Medium,0,0,0,1,0,116,102,78,148,208,0
3,75,Male,106,171,37.4,Never,Moderate,Average,Low,0,0,1,0,0,171,92,109,105,290,1
4,34,Female,65,191,18.5,Current,Sedentary,Healthy,Low,1,1,0,0,0,164,67,108,116,220,1


In [6]:
ordinal_encoder1=OrdinalEncoder(categories=[["Male","Female"]])
df["Gender"] = ordinal_encoder1.fit_transform(df[["Gender"]])

ordinal_encoder2 = OrdinalEncoder(categories = [["Never", "Current", "Former"]])
df["Smoking"] = ordinal_encoder2.fit_transform(df[["Smoking"]])


ordinal_encoder3 = OrdinalEncoder(categories = [["Moderate", "Sedentary", "Active"]])
df["Physical_Activity"] = ordinal_encoder3.fit_transform(df[["Physical_Activity"]])

ordinal_encoder4 = OrdinalEncoder(categories = [["Unhealthy", "Average", "Healthy"]])
df["Diet"] = ordinal_encoder4.fit_transform(df[["Diet"]])

ordinal_encoder5 = OrdinalEncoder(categories = [["Low", "Medium", "High"]])
df["Stress_Level"] = ordinal_encoder5.fit_transform(df[["Stress_Level"]])

In [7]:
df.head()

,Age,Gender,Weight,Height,BMI,Smoking,Physical_Activity,Diet,Stress_Level,Hypertension,Diabetes,Hyperlipidemia,Family_History,Previous_Heart_Attack,Systolic_BP,Diastolic_BP,Heart_Rate,Blood_Sugar_Fasting,Cholesterol_Total,Heart_Disease
0,48,0.0,78,157,26.4,0.0,1.0,2.0,1.0,0,0,1,1,0,104,99,71,165,200,0
1,35,1.0,73,163,33.0,0.0,2.0,1.0,2.0,1,0,1,1,0,111,72,60,145,206,0
2,79,1.0,88,152,32.3,0.0,0.0,1.0,1.0,0,0,0,1,0,116,102,78,148,208,0
3,75,0.0,106,171,37.4,0.0,0.0,1.0,0.0,0,0,1,0,0,171,92,109,105,290,1
4,34,1.0,65,191,18.5,1.0,1.0,2.0,0.0,1,1,0,0,0,164,67,108,116,220,1


In [8]:
int_col = df.select_dtypes(int).columns.tolist()

for col in int_col:
    df[col] = df[col].astype(np.int8)


float_col = df.select_dtypes(float).columns.tolist()


for col in float_col:
    df[col] = df[col].astype(np.float32)

In [9]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 50000 entries, 0 to 49999
Data columns (total 20 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   Age                    50000 non-null  int8   
 1   Gender                 50000 non-null  float32
 2   Weight                 50000 non-null  int8   
 3   Height                 50000 non-null  int8   
 4   BMI                    50000 non-null  float32
 5   Smoking                50000 non-null  float32
 6   Physical_Activity      50000 non-null  float32
 7   Diet                   50000 non-null  float32
 8   Stress_Level           50000 non-null  float32
 9   Hypertension           50000 non-null  int8   
 10  Diabetes               50000 non-null  int8   
 11  Hyperlipidemia         50000 non-null  int8   
 12  Family_History         50000 non-null  int8   
 13  Previous_Heart_Attack  50000 non-null  int8   
 14  Systolic_BP            50000 non-null  int8   
 15  Di

In [10]:
X=df.drop(columns="Heart_Disease")
Y=df["Heart_Disease"]



In [ ]:
n_splits = [5, 10, 15, 20, 25, 30]

for split in n_splits:
    cv = StratifiedKFold(n_splits = split, shuffle = True, random_state = 42)

    accuries = []

    for train_index , test_index in cv.split(X,Y):
        X_train, X_test = X.iloc[train_index] , X.iloc[test_index]
        Y_train, Y_test = Y.iloc[train_index] , Y.iloc[test_index]

        scaler = StandardScaler()
        X_train = scaler.fit_transform(X_train)
        X_test = scaler.transform(X_test)

        model = XGBClassifier(
            n_estimators=500,
            learning_rate=0.1,
            max_depth=6,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric='logloss',
            random_state=42
        )
        
        model.fit(X_train, Y_train)
        
        Y_pred = model.predict(X_test)
        score = accuracy_score(Y_test, Y_pred)
    
        accuries.append(score)
    
    # print(accuries)
    print(np.mean(accuries))
    accuries.clear()
    




1.0
1.0
1.0
